In [2]:
import pandas as pd
import ast
import re
import difflib
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

from pathlib import Path
from tqdm import tqdm
import os

In [82]:
def remove_spaces_around_apostrophe_and_dash(text):
    text = text.replace(" ' ", "'")
    text = text.replace("' s", "'s")
    text = text.replace(" - ", "-")
    text = text.replace("- ", "-")
    text = text.replace(" / ", "/")
    text = text.replace("( ", "(")
    text = text.replace(" )", ")")
    text = text.replace("[ ", "[")
    text = text.replace(" ]", "]")
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
    

def extract_entities_to_columns(df, col, prefix=None):

    disease_col = []
    drug_col = []

    for row in df[col]:

        diseases = set()
        drugs = set()

        if isinstance(row, str):
            row = ast.literal_eval(row)

        for item in row:

            if len(item) != 4:
                continue

            start, end, label, entity_name = item

            if (
                not entity_name
                or len(entity_name) == 1
                or entity_name.startswith("####")
            ):
                continue

            entity_name = remove_spaces_around_apostrophe_and_dash(entity_name).lower()

            if label == "DISEASE":
                diseases.add(entity_name)
            elif label == "DRUG":
                drugs.add(entity_name)

        disease_col.append("|".join(sorted(diseases)))
        drug_col.append("|".join(sorted(drugs)))

    # Build column names
    if prefix:
        disease_name = f"{prefix}_DISEASE_entities"
        drug_name = f"{prefix}_DRUG_entities"
    else:
        disease_name = "DISEASE_entities"
        drug_name = "DRUG_entities"

    df[disease_name] = disease_col
    df[drug_name] = drug_col

    return df

In [84]:
def partial_match_simple(predicted, target):
    predicted_parts = predicted.lower().split()
    target_parts = target.lower().split()
    return any(pred_part in target_part or target_part in pred_part for pred_part in predicted_parts for target_part in target_parts)

# Function to check for partial matches using difflib
def partial_match(predicted, target, cutoff=0.6):
    # Split the conditions into individual terms
    predicted_terms = predicted.split('|')
    target_terms = target.split('|')
    
    # Check for matches across all terms
    matches = []
    for pred in predicted_terms:
        # Use difflib to find close matches, with a cutoff for match quality
        match_found = any(difflib.get_close_matches(pred, target_terms, n=1, cutoff=cutoff))
        matches.append(match_found)
    
    # Return True if any match is found
    return any(matches)

def handle_empty_arrays(entity, model, target_entities, predicted_entities):

    def entities_are_empty(entities):
        return (
            entities is None
            or len(entities) == 0
            or all(str(e).strip() == "" for e in entities)
        )

    target_is_empty = entities_are_empty(target_entities)
    predicted_is_empty = entities_are_empty(predicted_entities)

    index_names = [
        f'target_{entity}_annotations_exact_{model}',
        f'predicted_{entity}_annotations_exact_{model}',
        f'target_{entity}_annotations_partial_{model}',
        f'predicted_{entity}_annotations_partial_{model}'
    ]

    # Case 1: both empty
    if target_is_empty and predicted_is_empty:
        return pd.Series([[0], [0], [0], [0]], index=index_names)

    # Case 2: target empty but predictions exist
    if target_is_empty:
        n_predictions = len(predicted_entities)
        return pd.Series(
            [[0]*n_predictions, [1]*n_predictions, [0]*n_predictions, [1]*n_predictions],
            index=index_names
        )

    # Case 3: predictions empty but target exists
    n_targets = len(target_entities)
    return pd.Series(
        [[1]*n_targets, [0]*n_targets, [1]*n_targets, [0]*n_targets],
        index=index_names
    )

# Function to compute exact and partial annotations for predicted and target conditions
def compute_annotations(row, entity, model):

    target_col = f'unique_{entity.replace("_v1","").replace("_v2","")}_target'
    model_col = f'unique_{entity}_{model}'
    
    # --- Parse and lowercase ---
    target_entities = (
        [] if pd.isna(row[target_col]) or str(row[target_col]).strip() == ""
        else [e.strip().lower() for e in str(row[target_col]).split('|') if len(e.strip()) > 2]
    )

    predicted_entities = (
        [] if pd.isna(row[model_col]) or str(row[model_col]).strip() == ""
        else [e.strip().lower() for e in str(row[model_col]).split('|') if len(e.strip()) > 2]
    )

    if not target_entities or not predicted_entities:
        #print("Handling empty")
        return handle_empty_arrays(entity, model, target_entities, predicted_entities)

    # Create a set of all unique entities from both target and predicted for consistent indexing
    all_entities = set(target_entities + predicted_entities)

    # Target annotations (exact)
    target_annotations_exact = [1 if entity in target_entities else 0 for entity in all_entities]

    # Predicted annotations (exact)
    predicted_annotations_exact = [1 if entity in predicted_entities else 0 for entity in all_entities]

    # Target annotations (partial)
    target_annotations_partial = [1 if any(partial_match(entity, target_ent) for target_ent in target_entities) else 0 for entity in all_entities]

    # Predicted annotations (partial)
    predicted_annotations_partial = [1 if any(partial_match(entity, pred) for pred in predicted_entities) else 0 for entity in all_entities]

    return pd.Series([
        target_annotations_exact,
        predicted_annotations_exact,
        target_annotations_partial,
        predicted_annotations_partial
    ], index=[
        f'target_{entity}_annotations_exact_{model}',
        f'predicted_{entity}_annotations_exact_{model}',
        f'target_{entity}_annotations_partial_{model}',
        f'predicted_{entity}_annotations_partial_{model}'
    ])



In [86]:
def flatten_column_arrays(column):
    # Prepare an array to hold the actual arrays/lists
    prepared_arrays = []
    
    for item in column:
        if isinstance(item, str):
            try:
                # Safely evaluate the string to see if it represents a list
                evaluated_item = ast.literal_eval(item)
                if isinstance(evaluated_item, list):
                    prepared_arrays.append(np.array(evaluated_item))
            except (SyntaxError, ValueError):
                # If evaluation fails or it's not a list, skip or handle non-list string
                print("Not possible to flatten ", item)
                continue
        elif isinstance(item, list):
            prepared_arrays.append(np.array(item))
        elif isinstance(item, np.ndarray):
            prepared_test_arrays.append(item)
    
    if prepared_arrays:
        # Concatenate all arrays in the list and then flatten the result
        return np.concatenate(prepared_arrays).flatten()
    else:
        return np.array([])  # Return an empty array if no valid arrays were found

def round_tuple(x, decimals=2):
    """
    Recursively round elements of a tuple.
    If x is a scalar (int/float), return the rounded value directly.
    """
    if isinstance(x, (int, float)):
        return round(x, decimals)
    elif isinstance(x, tuple):
        return tuple(round_tuple(item, decimals) for item in x)
    else:
        raise TypeError(f"Unsupported type {type(x)} for round_tuple")

def calculate_exact_and_partial_match_scores(annotations_df, entities, models, round_to=2):
    exact_matches = {}
    partial_matches = {}
    scores = {}
    for entity in entities:
        exact_matches[entity] = {}
        partial_matches[entity] = {}
        scores[entity] = {}
    
        for model in models:
            if (entity != 'condition' and entity != 'drug') and 'regex' in model:
               continue  
            #try:
            # Check if all columns are present in the DataFrame
            # Columns to check
            columns_to_check = [f'target_{entity}_annotations_exact_{model}',
                                f'predicted_{entity}_annotations_exact_{model}',
                                f'target_{entity}_annotations_partial_{model}',
                                f'predicted_{entity}_annotations_partial_{model}']
            
            # Check if all columns are present in the DataFrame
            missing_columns = [col for col in columns_to_check if col not in annotations_df.columns]
            if not missing_columns:
                flattened_data_target_exact = flatten_column_arrays(annotations_df[f'target_{entity}_annotations_exact_{model}'])
                flattened_data_model_exact = flatten_column_arrays(annotations_df[f'predicted_{entity}_annotations_exact_{model}'])
                flattened_data_target_partial = flatten_column_arrays(annotations_df[f'target_{entity}_annotations_partial_{model}'])
                flattened_data_model_partial = flatten_column_arrays(annotations_df[f'predicted_{entity}_annotations_partial_{model}'])
            else:
                # Handle the case where one or more columns are missing
                print("The following required columns are missing in the DataFrame:")
                for col in missing_columns:
                    print(col)
    
            flattened_data_target_exact = flatten_column_arrays(annotations_df.get(f'target_{entity}_annotations_exact_{model}', pd.Series()))
            flattened_data_model_exact = flatten_column_arrays(annotations_df.get(f'predicted_{entity}_annotations_exact_{model}', pd.Series()))
            flattened_data_target_partial = flatten_column_arrays(annotations_df.get(f'target_{entity}_annotations_partial_{model}', pd.Series()))
            flattened_data_model_partial = flatten_column_arrays(annotations_df.get(f'predicted_{entity}_annotations_partial_{model}', pd.Series()))

            # Calculate and store F1 scores
            exact_f1 = f1_score(flattened_data_target_exact, flattened_data_model_exact, average='binary')
            partial_f1 = f1_score(flattened_data_target_partial, flattened_data_model_partial, average='binary')

            exact_precision = precision_score(flattened_data_target_exact, flattened_data_model_exact)
            partial_precision = precision_score(flattened_data_target_partial, flattened_data_model_partial)

            exact_recall = recall_score(flattened_data_target_exact, flattened_data_model_exact)
            partial_recall = recall_score(flattened_data_target_partial, flattened_data_model_partial)

            exact_matches[entity][model] = round_tuple(exact_f1,round_to)
            partial_matches[entity][model] = round_tuple(partial_f1,round_to)

            scores[entity][model] = {
                "exact": {
                    "precision": round_tuple(exact_precision, round_to),
                    "recall": round_tuple(exact_recall, round_to),
                    "f1": round_tuple(exact_f1, round_to),
                },
                "partial": {
                    "precision": round_tuple(partial_precision, round_to),
                    "recall": round_tuple(partial_recall, round_to),
                    "f1": round_tuple(partial_f1, round_to),
                }
            }
    
           # except Exception as e:
            #    print(f"An error occurred for {model} in {entity}: {e}")
    return exact_matches, partial_matches, scores

In [132]:
import pandas as pd

ENTITY_TYPES = ["DISEASE", "DRUG"]

def load_pred_fold(fold):
    df = pd.read_csv(
        f"./model_predictions/drug_disease/preclinie_test_set/"
        f"test_annotated_BioLinkBERT-base_tuples_20251001_fold_{fold}.csv"
    )[["doc_id", "ner_prediction_BioLinkBERT-base_normalized"]]

    df = extract_entities_to_columns(
        df,
        "ner_prediction_BioLinkBERT-base_normalized",
        prefix="pred"
    )
    return df


def load_target_fold(fold):
    df = pd.read_csv(
        f"./data/finetuning_drug_disease/k_fold_splits/test_fold_{fold}.csv"
    )[["doc_id", "annotations_tuples", "Text"]]

    df = extract_entities_to_columns(
        df,
        "annotations_tuples",
        prefix="target"
    )
    return df


def merge_pred_target(pred_df, target_df):
    df_merged = pred_df[["doc_id", "pred_DISEASE_entities", "pred_DRUG_entities"]].merge(
        target_df[["doc_id", "target_DISEASE_entities", "target_DRUG_entities"]],
        on="doc_id",
        how="left"
    )
    return df_merged


def rename_to_unique(df_merged, model_name="biolinkbert", entity_types=ENTITY_TYPES):
    rename_dict = {}
    for entity in entity_types:
        rename_dict[f"target_{entity}_entities"] = f"unique_{entity}_target"
        rename_dict[f"pred_{entity}_entities"] = f"unique_{entity}_{model_name}"

    return df_merged.rename(columns=rename_dict)


def add_annotation_columns(df, model_name="biolinkbert", entity_types=ENTITY_TYPES):
    models = [model_name]
    out = df.copy()

    for entity in entity_types:
        for m in models:
            annotations_cols = [
                f"target_{entity}_annotations_exact_{m}",
                f"predicted_{entity}_annotations_exact_{m}",
                f"target_{entity}_annotations_partial_{m}",
                f"predicted_{entity}_annotations_partial_{m}",
            ]

            # same logic; expand safely into 4 columns
            out[annotations_cols] = out.apply(
                lambda row: compute_annotations(row, entity, m),
                axis=1,
                result_type="expand"
            )

    return out


def score_predictions(predictions_subset, model_name="biolinkbert", entity_types=ENTITY_TYPES):
    models = [model_name]
    exact_matches, partial_matches, scores = calculate_exact_and_partial_match_scores(
        predictions_subset, entity_types, models
    )
    return exact_matches, partial_matches, scores


def run_one_fold(fold, model_name="biolinkbert"):
    pred_df = load_pred_fold(fold)
    target_df = load_target_fold(fold)

    merged = merge_pred_target(pred_df, target_df)
    merged = rename_to_unique(merged, model_name=model_name)
    merged = merged[~merged["doc_id"].str.contains("method", na=False)]
    print(f"Evaluating fold {fold}: {len(merged)} rows.")

    predictions_subset = add_annotation_columns(merged, model_name=model_name)

    exact_matches, partial_matches, scores = score_predictions(
        predictions_subset, model_name=model_name
    )
    return scores, predictions_subset

In [134]:
fold = 0
pred_df = load_pred_fold(fold)
target_df = load_target_fold(fold)

merged = merge_pred_target(pred_df, target_df)
merged = merged[~merged["doc_id"].str.contains("method", na=False)]
merged_unique = rename_to_unique(merged, model_name="biolinkbert")

preds_with_ann = add_annotation_columns(merged_unique, model_name="biolinkbert")

exact_matches, partial_matches, scores = score_predictions(preds_with_ann, model_name="biolinkbert")
scores

{'DISEASE': {'biolinkbert': {'exact': {'precision': 0.66,
    'recall': 0.73,
    'f1': 0.7},
   'partial': {'precision': 0.77, 'recall': 0.84, 'f1': 0.8}}},
 'DRUG': {'biolinkbert': {'exact': {'precision': 0.6,
    'recall': 0.58,
    'f1': 0.59},
   'partial': {'precision': 0.88, 'recall': 0.86, 'f1': 0.87}}}}

In [136]:
row = merged_unique.iloc[3]
row

doc_id                                         My_pdf810_title^abstract
unique_DISEASE_biolinkbert    aneurysm|aneurysms|intracranial aneurysms
unique_DRUG_biolinkbert                                                
unique_DISEASE_target                  aneurysms|intracranial aneurysms
unique_DRUG_target                                                     
Name: 6, dtype: object

In [138]:
compute_annotations(row, entity="DRUG", model="biolinkbert")

target_DRUG_annotations_exact_biolinkbert         [0]
predicted_DRUG_annotations_exact_biolinkbert      [0]
target_DRUG_annotations_partial_biolinkbert       [0]
predicted_DRUG_annotations_partial_biolinkbert    [0]
dtype: object

In [148]:
preds_with_ann.head(10)

,doc_id,unique_DISEASE_biolinkbert,unique_DRUG_biolinkbert,unique_DISEASE_target,unique_DRUG_target,target_DISEASE_annotations_exact_biolinkbert,predicted_DISEASE_annotations_exact_biolinkbert,target_DISEASE_annotations_partial_biolinkbert,predicted_DISEASE_annotations_partial_biolinkbert,target_DRUG_annotations_exact_biolinkbert,predicted_DRUG_annotations_exact_biolinkbert,target_DRUG_annotations_partial_biolinkbert,predicted_DRUG_annotations_partial_biolinkbert
0,My_pdf953_title^abstract,neuropathic pain,pregabalin,neuropathic pain|peripheral neuropathy,chloroform|pregabalin,"[1, 1]","[0, 1]","[1, 1]","[0, 1]","[1, 1]","[1, 0]","[1, 1]","[1, 0]"
2,My_pdf329new_title^abstract,epilepsy|schizophrenia,5-oh-indole|acid|allosteric modulator|choline|...,epilepsy|schizophrenia,5-oh-indole|bicuculline|cd(2+)|choline|gaba(a)...,"[1, 1]","[1, 1]","[1, 1]","[1, 1]","[1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1]","[0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0]","[1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1]","[0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1]"
4,My_pdf378new_title^abstract,neuropathic pain|persistent pain,12-lipoxygenase inhibitor|baicalein,neuropathic pain,12-lipoxygenase inhibitor|baicalein,"[0, 1]","[1, 1]","[0, 1]","[1, 1]","[1, 1]","[1, 1]","[1, 1]","[1, 1]"
6,My_pdf810_title^abstract,aneurysm|aneurysms|intracranial aneurysms,,aneurysms|intracranial aneurysms,,"[0, 1, 1]","[1, 1, 1]","[1, 1, 1]","[1, 1, 1]",[0],[0],[0],[0]
8,My_pdf680_title^abstract,traumatic spinal cord injury,,traumatic spinal cord injury,,[1],[1],[1],[1],[0],[0],[0],[0]
10,My_pdf509new_title^abstract,glioma|intracranial gliomas|malignant glioma,anti-ctla -|anti-ctla-4 monoclonal antibodies|...,intracranial gliomas|intracranial tumors|malig...,anti-ctla-4 monoclonal antibodies|ctla-4 block...,"[0, 1, 1, 1]","[1, 1, 1, 0]","[0, 1, 1, 1]","[1, 1, 1, 1]","[1, 1, 0, 1, 0]","[0, 1, 1, 0, 1]","[1, 1, 1, 1, 0]","[0, 1, 1, 0, 1]"
12,My_pdf27new_title^abstract,brain edema|haemophilus influenzae type b meni...,ceftriaxone|dexamethasone,meningitis,ceftriaxone|dexamethasone,"[0, 1, 0, 0]","[1, 1, 1, 1]","[0, 1, 0, 1]","[1, 1, 1, 1]","[1, 1]","[1, 1]","[1, 1]","[1, 1]"
14,My_pdf233new_title^abstract,,,congenital deafness,,[1],[0],[1],[0],[0],[0],[0],[0]
16,My_pdf133new_title^abstract,cerebral ischemia|ischemic injury,ia|ilexonin a,cerebral ischemia and reperfusion|cerebral isc...,ia|ilexonin a,"[1, 1, 1, 0]","[0, 1, 0, 1]","[1, 1, 1, 1]","[1, 1, 1, 1]",[1],[1],[1],[1]
18,My_pdf162new_title^abstract,sci|spinal cord injury,probucol,sci|spinal cord injury,probucol,"[1, 1]","[1, 1]","[1, 1]","[1, 1]",[1],[1],[1],[1]


In [142]:
all_scores = {}
for fold in range(10):
    scores, _ = run_one_fold(fold, model_name="biolinkbert")
    all_scores[fold] = scores

Evaluating fold 0: 81 rows.
Evaluating fold 1: 81 rows.
Evaluating fold 2: 79 rows.
Evaluating fold 3: 79 rows.
Evaluating fold 4: 79 rows.
Evaluating fold 5: 79 rows.
Evaluating fold 6: 79 rows.
Evaluating fold 7: 79 rows.
Evaluating fold 8: 79 rows.
Evaluating fold 9: 78 rows.


In [143]:
def flatten_scores(all_scores, model="biolinkbert"):
    rows = []
    for fold, s in all_scores.items():
        for entity in ["DISEASE", "DRUG"]:
            for match in ["exact", "partial"]:
                d = s[entity][model][match]
                rows.append({
                    "fold": fold,
                    "entity": entity,
                    "match": match,
                    "precision": d["precision"],
                    "recall": d["recall"],
                    "f1": d["f1"],
                })
    return pd.DataFrame(rows)

scores_df = flatten_scores(all_scores, model="biolinkbert")
summary = scores_df.groupby(["entity", "match"])[["precision","recall","f1"]].agg(["mean","std"]).reset_index()

scores_df, summary

(    fold   entity    match  precision  recall    f1
 0      0  DISEASE    exact       0.66    0.73  0.70
 1      0  DISEASE  partial       0.77    0.84  0.80
 2      0     DRUG    exact       0.60    0.58  0.59
 3      0     DRUG  partial       0.88    0.86  0.87
 4      1  DISEASE    exact       0.58    0.69  0.63
 5      1  DISEASE  partial       0.77    0.86  0.81
 6      1     DRUG    exact       0.55    0.69  0.61
 7      1     DRUG  partial       0.73    0.86  0.79
 8      2  DISEASE    exact       0.67    0.74  0.70
 9      2  DISEASE  partial       0.85    0.86  0.85
 10     2     DRUG    exact       0.60    0.76  0.67
 11     2     DRUG  partial       0.79    0.95  0.86
 12     3  DISEASE    exact       0.56    0.74  0.64
 13     3  DISEASE  partial       0.74    0.89  0.81
 14     3     DRUG    exact       0.51    0.60  0.55
 15     3     DRUG  partial       0.73    0.80  0.76
 16     4  DISEASE    exact       0.68    0.73  0.70
 17     4  DISEASE  partial       0.81    0.82